# HairFastGAN — ทดลองทรงผมและสีผมบน Google Colab
คู่มือภาษาไทย • ใช้น้ำหนัก pretrained ของผู้วิจัย • ภาพเข้า 3 รูป → ผลลัพธ์ PNG 1024 × 1024

**เริ่มต้น:** เลือก Runtime → Change runtime type → **T4 GPU** แล้วรันเซลล์จากบนลงล่าง การติดตั้งและดาวน์โหลดครั้งแรกใช้เวลาหลายนาทีและพื้นที่หลาย GB ต้องเชื่อมต่ออินเทอร์เน็ต ไม่ต้องใส่ API key

Notebook นี้ใช้ Python 3.10 แยกจาก kernel ของ Colab และ PyTorch 1.13.1 ตามโครงการเดิม จึงไม่ต้องเปลี่ยน Python ของ Colab หรือ restart kernel เป้าหมายคือ T4; ไม่รองรับ GPU รุ่นใหม่ทุกสถาปัตยกรรมโดยอัตโนมัติ

**สถานะการตรวจสอบ:** ตรวจโครงสร้าง notebook, syntax, API และตำแหน่งน้ำหนักเทียบกับต้นฉบับแล้ว แต่ยังไม่ได้รันติดตั้งและ inference จริงบน Colab GPU ในสภาพแวดล้อมที่จัดทำไฟล์นี้ จึงยังรับรองการรันจริงทุกเซลล์ไม่ได้ หากเกิดข้อผิดพลาดให้เก็บ log ตามท้าย notebook


## 1. งานวิจัยทำอะไร
อ้างอิง Nikolaev และคณะ, *HairFastGAN: Realistic and Robust Hair Transfer with a Fast Encoder-Based Approach*, arXiv:2404.01094v3 (25 พฤษภาคม 2024)

| ภาพเข้า | หน้าที่ |
|---|---|
| Face | ใบหน้าของผู้ใช้ที่ต้องการรักษาเอกลักษณ์ |
| Shape | ภาพบุคคลที่มีทรงผมต้องการ |
| Color | ภาพบุคคลที่มีสีผมต้องการ |

รูป Color ต้องมีใบหน้าและเส้นผมให้โมเดลตรวจจับ ไม่ใช่ภาพช่องสีล้วน เลือกรูปชัด เห็นหน้าและเส้นผมเต็ม ใช้คนเดียวต่อภาพ มุมใกล้เคียงกันช่วยให้ผลลัพธ์ดีขึ้น

ลำดับหลักจากบทความและโค้ด:
1. **Embedding:** e4e และ FeatureStyle Encoder แปลงภาพเป็น latent พร้อมแยกบริเวณหน้า/ผมด้วย BiSeNet
2. **Pose alignment:** Rotate Encoder ปรับตัวแทนทรงผมให้เข้ากับมุมหน้าภาพผู้ใช้
3. **Shape alignment:** Shape Adaptor และ SEAN ช่วยจัดหน้ากากและข้อมูลทรงผม แล้วผสม feature ของ StyleGAN
4. **Color alignment:** encoder ที่ใช้ CLIP ช่วยถ่ายทอดสีผมจากภาพ Color
5. **Refinement alignment:** encoder ปรับรายละเอียดใบหน้าและฉากหลัง ก่อน StyleGAN2 สร้างภาพสุดท้าย

**StyleGAN2** เป็นตัวสร้างภาพจาก latent: S ควบคุมลักษณะภาพ และ F เก็บข้อมูลเชิงพื้นที่ให้แก้เฉพาะส่วนได้ละเอียดขึ้น โมเดลไม่ได้แปะผมลงบนภาพตรง ๆ จึงยังอาจเปลี่ยนรายละเอียดหน้าได้

**Pretrained** หมายถึงโหลดน้ำหนักที่ผู้วิจัยฝึกไว้แล้วมา inference; notebook นี้ไม่มีการ train หรือ fine-tune ส่วน HairFastGAN ไม่มี diffusion / upscaler; ส่วน 8 เพิ่ม Realistic Vision V5.1 แบบเลือกใช้งาน ผลเป็นภาพครอปที่ align แล้ว ไม่ใช่ภาพต้นฉบับเต็มตัว


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, time
BASE = Path('/content/hairfast_colab')
BASE.mkdir(parents=True, exist_ok=True)
REPO = BASE / 'HairFastGAN'
PY = BASE / 'venv/bin/python'
CODE_REV = '49e98019c9243d1991c19d45764fc3340a498de2'
WEIGHTS_REV = '47855f8ea72b02294649385211a1ae1fd7dcb58e'

def run(args, cwd=None, log_name='setup.log'):
    # แสดง log แบบต่อเนื่องและเก็บไฟล์ไว้ตรวจย้อนหลัง
    with (BASE / log_name).open('a', encoding='utf-8') as log:
        p = subprocess.Popen([str(x) for x in args], cwd=cwd, env={**os.environ, 'PYTHONUNBUFFERED':'1', 'WANDB_MODE':'disabled'},
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end=''); log.write(line); log.flush()
        rc = p.wait()
    if rc:
        raise RuntimeError(f'คำสั่งล้มเหลว (exit {rc}); ดู {BASE / log_name} แล้วแก้ก่อนรันเซลล์ถัดไป')

run(['nvidia-smi'])
print('พร้อมตรวจ GPU — แนะนำ T4 หากไม่ได้เปิด GPU ให้เปลี่ยน runtime ก่อน')


## 2. ติดตั้ง Python และ dependencies
ใช้รุ่นเดิมเป็นหลัก เพิ่ม `lpips` ที่ source import แต่ requirements เดิมไม่ได้ระบุไว้ และเปลี่ยน dlib เป็น 19.24.6 ซึ่งอาจต้องคอมไพล์ ช่วงนี้อาจนาน ห้ามข้ามหากยังติดตั้งไม่สำเร็จ


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.6.17'])
run(['apt-get', 'update', '-qq'])
run(['apt-get', 'install', '-y', '-qq', 'build-essential', 'cmake'])
run([sys.executable, '-m', 'uv', 'python', 'install', '3.10'])
if not PY.exists():
    run([sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '--seed', str(PY.parent.parent)])
if not (REPO / '.git').exists():
    run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', str(REPO)])
run(['git', 'checkout', CODE_REV], cwd=REPO)
# ใช้ PyTorch CUDA wheel ที่จับคู่กับ torchvision รุ่นเดียวกัน
run([PY, '-m', 'pip', 'install', 'pip==24.3.1', 'setuptools==69.5.1', 'wheel'])
run([PY, '-m', 'pip', 'install', 'torch==1.13.1+cu117', 'torchvision==0.14.1+cu117',
     '--extra-index-url', 'https://download.pytorch.org/whl/cu117'])
excluded = ('torch==', 'torchvision==', 'dlib==')
requirements = [s for s in (REPO/'requirements.txt').read_text().splitlines() if s and not s.startswith(excluded)]
requirements += ['dlib==19.24.6', 'lpips==0.1.4', 'huggingface_hub==0.25.2']
req_file = BASE/'requirements_colab.txt'
req_file.write_text('\n'.join(requirements)+'\n')
run([PY, '-m', 'pip', 'install', '-r', req_file])
run([PY, '-m', 'pip', 'check'])
run([PY, '-c', "import torch, torchvision, dlib, lpips; print('torch',torch.__version__,'vision',torchvision.__version__); assert torch.cuda.is_available(), 'GPU ไม่พร้อม'; print(torch.cuda.get_device_name(0)); print((torch.ones(1,device='cuda')+1).item())"])


## 3. ดาวน์โหลด pretrained ทั้งชุดและตรวจไฟล์
ดาวน์โหลดจาก Hugging Face ทางการโดยล็อก revision เพื่อให้ทำซ้ำได้ ใช้ snapshot download แทน Git LFS และตรวจว่าไฟล์ไม่ใช่ LFS pointer ถ้าเน็ตหลุดให้รันเซลล์นี้ซ้ำได้


In [ ]:
download_script = r"""
from huggingface_hub import snapshot_download
from pathlib import Path
import json
repo = Path(__file__).parent / 'HairFastGAN'
snapshot_download(repo_id='AIRI-Institute/HairFastGAN',
 revision='47855f8ea72b02294649385211a1ae1fd7dcb58e',
 allow_patterns=['pretrained_models/*'], local_dir=str(repo), max_workers=2)
required = [
 'StyleGAN/ffhq.pt','StyleGAN/ffhq_PCA.npz',
 'encoder4editing/e4e_ffhq_encode.pt',
 'FeatureStyleEncoder/143_enc.pth','FeatureStyleEncoder/psp_ffhq_encode.pt',
 'FeatureStyleEncoder/backbone.pth','FeatureStyleEncoder/79999_iter.pth',
 'BiSeNet/face_parsing_79999_iter.pth','ShapeAdaptor/mask_generator.pth',
 'ShapeAdaptor/shape_predictor_68_face_landmarks.dat',
 'Rotate/rotate_best.pth','Blending/checkpoint.pth',
 'PostProcess/pp_model.pth','PostProcess/latent_avg.pt','ArcFace/backbone_ir50.pth',
 'sean_checkpoints/CelebA-HQ_pretrained/latest_net_G.pth']
for name in required:
 p=repo/'pretrained_models'/name
 assert p.is_file() and p.stat().st_size > 128, f'Missing/empty: {p}'
 with p.open('rb') as f:
  assert not f.read(128).startswith(b'version https://git-lfs'), f'LFS pointer: {p}'
 print(f'OK {name}: {p.stat().st_size/1024**2:.1f} MiB')
(repo.parent/'weights_manifest.json').write_text(json.dumps({
 'repo':'AIRI-Institute/HairFastGAN',
 'revision':'47855f8ea72b02294649385211a1ae1fd7dcb58e',
 'files':{str(p.relative_to(repo)):p.stat().st_size for p in (repo/'pretrained_models').rglob('*') if p.is_file()}
}, indent=2))
"""
(BASE/'download_weights.py').write_text(download_script)
run([PY, BASE/'download_weights.py'], log_name='weights.log')


## 4. ใช้ StyleGAN operators แบบ PyTorch เพื่อเลี่ยงการคอมไพล์ CUDA
ใช้ `upfirdn2d_native` ที่มีอยู่ในต้นฉบับ และคำนวณ bias + LeakyReLU ด้วย PyTorch บน GPU น้ำหนักและโมดูลหลักคงเดิม แต่ความเร็วและความต่างเชิงตัวเลขอาจต่างจาก CUDA kernel ของบทความ จึงไม่ควรนำเวลาจาก notebook ไปอ้างว่าเป็นเวลาทางการ


In [ ]:
import ast
# แก้เฉพาะไฟล์ op ในสำเนา Colab และสำรองไฟล์เดิมทุกไฟล์
for op_dir in sorted(REPO.rglob('stylegan2/op')):
    fused_path = op_dir/'fused_act.py'
    up_path = op_dir/'upfirdn2d.py'
    if not fused_path.exists() or not up_path.exists():
        continue
    for path in (fused_path, up_path):
        backup = path.with_suffix('.py.original')
        if not backup.exists():
            backup.write_text(path.read_text())
    fused_path.write_text("""import torch
from torch import nn
from torch.nn import functional as F

def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2**0.5):
    if bias is not None:
        input = input + bias.view(1, -1, *([1] * (input.ndim-2)))
    return F.leaky_relu(input, negative_slope=negative_slope) * scale

class FusedLeakyReLU(nn.Module):
    def __init__(self, channel, negative_slope=0.2, scale=2**0.5):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(channel))
        self.negative_slope, self.scale = negative_slope, scale
    def forward(self, input):
        return fused_leaky_relu(input, self.bias, self.negative_slope, self.scale)
""")
    original = up_path.with_suffix('.py.original').read_text()
    tree = ast.parse(original)
    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native')
    native = ast.get_source_segment(original, node)
    up_path.write_text('import torch\nfrom torch.nn import functional as F\n\n' + native + """

def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):
    return upfirdn2d_native(input, kernel, up, up, down, down,
                           pad[0], pad[1], pad[0], pad[1])
""")
    print('Patched:', op_dir.relative_to(REPO))
run([PY, '-c', 'from hair_swap import HairFast, get_parser; print("Import HairFast OK")'], cwd=REPO)


## 5. อัปโหลดภาพทีละหน้าที่
แต่ละปุ่มเลือก **1 ภาพ** รองรับ JPEG / PNG / WebP ที่ PIL เปิดได้ รูปจะถูกแก้ EXIF orientation และแปลง RGB PNG ก่อนส่งเข้าโมเดล ภาพทั้งสามต้องเห็นใบหน้า หากมีหลายหน้า โค้ดทางการจะเลือกหน้าที่ใหญ่ที่สุด


In [ ]:
from google.colab import files
from PIL import Image, ImageOps
from IPython.display import display
import io
INPUTS = BASE/'uploads'
INPUTS.mkdir(exist_ok=True)
paths = {}
for role, label in [('face','รูปใบหน้าของคุณ'), ('shape','รูปบุคคลที่มีทรงผมต้องการ'), ('color','รูปบุคคลที่มีสีผมต้องการ')]:
    print(f'อัปโหลด {label} (เลือก 1 ภาพ)')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('ต้องเลือก 1 ภาพต่อขั้นตอน กรุณารันเซลล์นี้ใหม่')
    original_name, data = next(iter(uploaded.items()))
    try:
        with Image.open(io.BytesIO(data)) as im:
            image = ImageOps.exif_transpose(im).convert('RGB')
    except Exception as e:
        raise ValueError(f'เปิดภาพ {original_name} ไม่ได้ กรุณาใช้ JPEG หรือ PNG') from e
    # ลดเฉพาะภาพใหญ่มากก่อนตรวจจับหน้า ไม่ยืดภาพให้เป็นสี่เหลี่ยม
    image.thumbnail((2048,2048), Image.Resampling.LANCZOS)
    path = INPUTS/f'{role}.png'
    image.save(path)
    paths[role] = str(path)
    preview=image.copy(); preview.thumbnail((350,350)); display(preview)
(BASE/'inputs.json').write_text(json.dumps(paths))
print('อัปโหลดครบ 3 ภาพแล้ว')


## 6. โหลดโมเดลและสร้างภาพ
เซลล์นี้จะโหลดโมเดล pretrained แล้วทำ alignment และ inference ครบทุกขั้นตอน โดยใช้ API `HairFast.swap(..., align=True)` ของโครงการโดยตรง ครั้งแรกอาจดาวน์โหลด CLIP เพิ่ม ผลที่คืนเมื่อ `align=True` เป็น tuple จึงต้องแยกภาพผลลัพธ์ออกจากภาพที่ครอป

สามารถอัปโหลดชุดใหม่ในข้อ 5 แล้วรันข้อนี้ซ้ำได้ แต่จะโหลดโมเดลใหม่ในแต่ละรอบเพื่อแยก environment ออกจาก Colab kernel เวลาจะแสดงแยกส่วนโหลดโมเดลและส่วนสร้างภาพ ซึ่งส่วนสร้างภาพรวม face alignment


In [ ]:
SEED = 3407 #@param {type:"integer"}
assert (BASE/'inputs.json').exists(), 'อัปโหลดภาพในข้อ 5 ก่อน'
run_id = time.strftime('%Y%m%d_%H%M%S', time.gmtime()) + '_' + str(time.time_ns())[-6:]
OUT = BASE/'results'/run_id

worker = "import argparse, json, time, sys, subprocess\nfrom pathlib import Path\nimport torch\nfrom torchvision.utils import save_image\nfrom hair_swap import HairFast, get_parser\n\np=argparse.ArgumentParser()\np.add_argument('--inputs', required=True)\np.add_argument('--out', required=True)\np.add_argument('--seed', type=int, default=3407)\na=p.parse_args()\nassert torch.cuda.is_available(), 'Please select a T4 GPU runtime'\ninputs=json.loads(Path(a.inputs).read_text())\nout=Path(a.out); out.mkdir(parents=True, exist_ok=True)\nopts=get_parser().parse_args([])\nopts.batch_size=1\nopts.device='cuda'\nprint('Loading pretrained models...', flush=True)\nt=time.perf_counter()\nmodel=HairFast(opts)\ntorch.cuda.synchronize()\nload_sec=time.perf_counter()-t\nprint(f'Model loaded: {load_sec:.2f} s', flush=True)\nt=time.perf_counter()\nwith torch.inference_mode():\n    result, face, shape, color = model.swap(inputs['face'],inputs['shape'],inputs['color'],align=True,seed=a.seed)\ntorch.cuda.synchronize()\ninference_sec=time.perf_counter()-t\nassert tuple(result.shape)==(3,1024,1024), f'Unexpected output shape: {result.shape}'\nassert torch.isfinite(result).all(), 'Result has NaN/Inf'\nsave_image(result.detach().cpu().clamp(0,1), out/'hairfast_result.png')\nfor role, im in zip(('face','shape','color'), (face,shape,color)):\n    # align_face returns float [0,1] tensors with the official default\n    save_image(im.detach().cpu().clamp(0,1), out/f'aligned_{role}.png')\nmetadata={\n 'seed':a.seed,'torch':torch.__version__, 'gpu':torch.cuda.get_device_name(0),\n 'code_revision':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),\n 'weights_revision':'47855f8ea72b02294649385211a1ae1fd7dcb58e',\n 'operators':'PyTorch native, no compiled CUDA extension',\n 'model_load_seconds':load_sec,'alignment_and_inference_seconds':inference_sec,\n 'size':[1024,1024],'inputs':inputs,\n}\n(out/'run_info.json').write_text(json.dumps(metadata,indent=2))\nprint(json.dumps(metadata,indent=2), flush=True)\n"
(REPO/'colab_infer.py').write_text(worker)
try:
    run([PY, REPO/'colab_infer.py', '--inputs', BASE/'inputs.json', '--out', OUT, '--seed', SEED],
        cwd=REPO, log_name='inference.log')
except RuntimeError:
    print('ถ้า CUDA out of memory: ลอง Disconnect and delete runtime แล้วเริ่มใหม่')
    print('ถ้า No faces detected: เปลี่ยนภาพให้เห็นหน้าชัด โดย 0=face, 1=shape, 2=color')
    raise
print('บันทึกผลที่', OUT)


## 7. ดูผลและดาวน์โหลดภาพ
ภาพอ้างอิงด้านล่างเป็นภาพที่ผ่าน alignment แล้ว เพื่อให้เทียบกับผลลัพธ์ได้ตรงกัน ดาวน์โหลดผลลัพธ์เป็น PNG แยกภาพโดยไม่ต้อง ZIP


In [ ]:
assert 'OUT' in globals() and (OUT/'hairfast_result.png').exists(), 'รันข้อ 6 ให้สำเร็จก่อน'
from PIL import ImageDraw
panel = Image.new('RGB',(1600,460),'white')
draw = ImageDraw.Draw(panel)
for i,(filename,label) in enumerate([
 ('aligned_face.png','Face'),('aligned_shape.png','Shape'),
 ('aligned_color.png','Color'),('hairfast_result.png','Result')]):
    with Image.open(OUT/filename) as im:
        im=im.convert('RGB').resize((390,390),Image.Resampling.LANCZOS)
        panel.paste(im,(i*400+5,45))
    draw.text((i*400+10,15),label,fill='black')
panel.save(OUT/'hairfast_comparison.png')
display(panel)
print('ภาพผลลัพธ์เต็มขนาด')
display(Image.open(OUT/'hairfast_result.png'))
files.download(str(OUT/'hairfast_result.png'))


In [ ]:
# รันแยกเมื่อต้องการดาวน์โหลดภาพเปรียบเทียบและข้อมูลการทดลอง
files.download(str(OUT/'hairfast_comparison.png'))
files.download(str(OUT/'run_info.json'))


## สรุปและข้อจำกัด
- ใช้ภาพหน้าของเรา + รูปบุคคลอ้างอิงทรงผม + รูปบุคคลอ้างอิงสีผม เพื่อสังเคราะห์ใบหน้าใหม่
- pretrained ทุกส่วนโหลดจากแหล่งทางการ ไม่ต้องเตรียมชุดฝึกและไม่ต้อง train ใหม่
- ผลลัพธ์ 1024 × 1024 ไม่ใช่ 4K/8K และการขยายภาพอย่างเดียวไม่เพิ่มรายละเอียดจริง
- การ align ครอปรูปตามใบหน้า จึงอาจตัดปลายผมยาวหรือพื้นที่รอบข้าง และยังไม่ได้วางผลกลับลงภาพเต็ม
- งานวิจัยรายงานความเร็วระดับต่ำกว่า 1 วินาทีบน V100 ในเงื่อนไขของผู้วิจัย ไม่รวมการติดตั้ง/ดาวน์โหลด และไม่ใช่คำรับประกันเวลา notebook นี้
- งานวิจัยระบุปัญหาการเติมพื้นที่เมื่อเปลี่ยนผมยาวเป็นสั้น และผมซับซ้อน เช่น เปีย หางม้า หรือริบบิ้น ผลขึ้นกับรูปและอาจคงใบหน้า/พื้นหลังได้ไม่สมบูรณ์

### ถ้ามีปัญหา
| อาการ | วิธีแก้ |
|---|---|
| `nvidia-smi` ไม่พบ / CUDA ไม่พร้อม | เปลี่ยน runtime เป็น T4 GPU แล้วเริ่มใหม่ |
| pip หรือดาวน์โหลดค้าง/ล้มเหลว | ดูข้อความท้าย log และรันเซลล์นั้นซ้ำ; ตรวจพื้นที่ว่าง/เน็ต |
| `No faces detected` | ตรวจภาพทั้งสาม ใช้ภาพหน้าเดียว ชัด ไม่หันข้างมาก; รูป Color ห้ามเป็นสีล้วน |
| `CUDA out of memory` | notebook ตั้ง batch_size=1 แล้ว; ล้าง runtime และเริ่มใหม่ หรือใช้ GPU หน่วยความจำมากกว่า |
| `no kernel image` | เลือก T4; PyTorch รุ่นนี้ไม่รองรับ GPU ใหม่ทุกชนิด |
| checkpoint หรือ PCA หาย | รันข้อ 3 ซ้ำก่อนสร้างโมเดล ห้ามปล่อยให้คำนวณ PCA ใหม่เอง |
| รูปไม่เหมือนเดิมหรือผมผิดรูป | ลองรูปชัด มุมใกล้กัน และผมไม่ถูกบัง; เป็นข้อจำกัดของโมเดล |

หากต้องการส่งข้อผิดพลาดให้ผู้ช่วย ให้แนบ log ด้านล่างและบอกเซลล์ที่ล้มเหลว ไม่ต้องส่งรูปส่วนตัวหากไม่จำเป็น


In [ ]:
# ดาวน์โหลด log ที่มีสำหรับตรวจปัญหา (ไม่บรรจุภาพ)
import zipfile
log_zip=BASE/'hairfast_diagnostics.zip'
with zipfile.ZipFile(log_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['setup.log','weights.log','inference.log','requirements_colab.txt','weights_manifest.json']:
        p=BASE/name
        if p.exists(): z.write(p,arcname=name)
files.download(str(log_zip))


## แหล่งอ้างอิงและสิ่งที่ปรับ
- บทความแนบ: arXiv:2404.01094v3, https://arxiv.org/abs/2404.01094
- Repository: https://github.com/AIRI-Institute/HairFastGAN
- โค้ดหลัก: `hair_swap.py`, `models/Embedding.py`, `models/Alignment.py`, `models/Blending.py`, `models/Net.py`
- น้ำหนัก: https://huggingface.co/AIRI-Institute/HairFastGAN
- ล็อก code commit `49e98019c9243d1991c19d45764fc3340a498de2` และ weights revision `47855f8ea72b02294649385211a1ae1fd7dcb58e`

ปรับเฉพาะสภาพแวดล้อม, dependency เสริม, การดาวน์โหลด, operators แบบ PyTorch, หน้ารับภาพ, การตรวจผลและบันทึก log ไม่มีการ fine-tune หรืออ้างว่าน้ำหนักเป็นโมเดลที่พัฒนาใหม่ ตรวจเงื่อนไขการใช้งานของโครงการและโมเดลประกอบเมื่อนำไปเผยแพร่ต่อ

```bibtex
@article{nikolaev2024hairfastgan,
  title={HairFastGAN: Realistic and Robust Hair Transfer with a Fast Encoder-Based Approach},
  author={Nikolaev, Maxim and Kuznetsov, Mikhail and Vetrov, Dmitry and Alanov, Aibek},
  journal={arXiv preprint arXiv:2404.01094},
  year={2024}
}
```


# 8. เพิ่ม Realistic Vision V5.1 หลัง HairFastGAN
ขั้นตอนเสริม **img2img** รับภาพ `hairfast_result.png` จากข้อ 6 โดยตรง ไม่ใช้รูปใบหน้าก่อนเปลี่ยนผมเป็นจุดเริ่มต้น

ใช้ strength=0.18, CFG=3.5 และผสมผลกลับกับภาพเดิม 65% เพื่อเริ่มจากการเปลี่ยนแปลงน้อย ช่วยลดโอกาสหน้า/ทรง/สีเปลี่ยน แต่ **ไม่ล็อกเอกลักษณ์ 100% และไม่รับประกันว่ารูปจะคมขึ้นทุกภาพ** prompt ไม่ใช่กลไกล็อกใบหน้า หากหน้าเปลี่ยนให้ลด strength และ blend alpha

ทำงานที่ขนาดภาพเดิม 1024 × 1024 ไม่ย่อก่อน diffusion และไม่เพิ่ม upscaler ส่วนนี้เป็นขั้นตอนที่เพิ่มเอง ไม่ใช่ส่วนหนึ่งของ HairFastGAN ในบทความ

ใช้ Python environment แยกอีกชุดเพื่อไม่ชนกับ PyTorch ของ HairFastGAN ทำงานบน GPU ทีละโมเดล เพราะขั้น HairFastGAN เดิมจบ subprocess และคืน VRAM แล้ว


In [ ]:
# 8.1 ติดตั้งเฉพาะส่วน diffusion: รันครั้งแรกครั้งเดียว
assert 'BASE' in globals() and 'run' in globals(), 'รันเซลล์ตั้งค่าข้อ 1 ก่อน'
SDPY = BASE/'sd_venv/bin/python'
if not SDPY.exists():
    run([sys.executable,'-m','uv','venv','--python','3.10','--seed',SDPY.parent.parent], log_name='diffusion_setup.log')
run([SDPY,'-m','pip','install','torch==2.5.1','torchvision==0.20.1',
     '--index-url','https://download.pytorch.org/whl/cu121'], log_name='diffusion_setup.log')
run([SDPY,'-m','pip','install','diffusers==0.32.2','transformers==4.46.3',
     'accelerate==1.2.1','huggingface-hub==0.27.1','safetensors==0.5.2',
     'numpy==1.26.4','Pillow==10.4.0','protobuf==4.25.3','sentencepiece==0.2.0'], log_name='diffusion_setup.log')
run([SDPY,'-m','pip','check'], log_name='diffusion_setup.log')
run([SDPY,'-c',"import torch,diffusers; assert torch.cuda.is_available(); print(torch.__version__,diffusers.__version__,torch.cuda.get_device_name(0))"],log_name='diffusion_setup.log')


## 8.2 ตั้งค่าการปรับภาพ
- `STRENGTH`: เริ่ม 0.18; ลดเป็น 0.10–0.15 หากหน้าหรือผมเปลี่ยน เพิ่มเป็น 0.22–0.25 หากต้องการปรับมากขึ้น แต่มีโอกาสเปลี่ยนลักษณะภาพมากขึ้น
- `BLEND_ALPHA`: สัดส่วนภาพ diffusion ในผลสุดท้าย 0=ภาพเดิม, 1=diffusion เต็มรูป; ค่าเริ่ม 0.65
- `STEPS`: 30 เป็นจำนวนขั้นที่ตั้งไว้ จำนวน denoising จริงถูกลดตาม strength (ค่าเริ่มประมาณ 5 ขั้น)
- `CFG`: 3.5 เพื่อไม่บังคับ prompt มากเกินไป
- กำหนด seed เพื่อช่วยทำซ้ำได้ภายใน environment เดียวกัน ไม่รับประกันผลเหมือนกันข้ามอุปกรณ์

เว้น `SOURCE_IMAGE` ว่างเพื่อใช้ผลรอบล่าสุดจาก `OUT`; หรือระบุ path PNG ที่มีอยู่ใน Colab ได้โดยตรง


In [ ]:
SOURCE_IMAGE = '' #@param {type:"string"}
STRENGTH = 0.18 #@param {type:"number"}
BLEND_ALPHA = 0.65 #@param {type:"number"}
STEPS = 30 #@param {type:"integer"}
CFG = 3.5 #@param {type:"number"}
SD_SEED = 3407 #@param {type:"integer"}
PROMPT = 'natural photograph, realistic skin texture, fine natural hair strands, subtle detail, natural lighting' #@param {type:"string"}
NEGATIVE_PROMPT = 'blurry, out of focus, plastic skin, waxy skin, oversharpened, halo, painting, cartoon, deformed face, distorted eyes, unnatural hair' #@param {type:"string"}
assert 0 < STRENGTH <= 0.5, 'แนะนำ strength ต่ำ; ต้องมากกว่า 0 และไม่เกิน 0.5'
assert 0 <= BLEND_ALPHA <= 1
assert 1 <= STEPS <= 100 and int(STEPS * STRENGTH) >= 1
assert 0 <= CFG <= 15
if SOURCE_IMAGE.strip():
    sd_input=Path(SOURCE_IMAGE).expanduser().resolve()
else:
    assert 'OUT' in globals(), 'รัน HairFastGAN ข้อ 6 หรือระบุ SOURCE_IMAGE'
    sd_input=OUT/'hairfast_result.png'
assert sd_input.is_file(), f'ไม่พบภาพ: {sd_input}'
with Image.open(sd_input) as im:
    assert im.width%8 == 0 and im.height%8 == 0, 'ขนาดภาพต้องหาร 8 ลงตัว; ภาพ HairFastGAN 1024 รองรับแล้ว'
SD_OUT=BASE/'diffusion_results'/(time.strftime('%Y%m%d_%H%M%S',time.gmtime())+'_'+str(time.time_ns())[-6:])
SD_OUT.mkdir(parents=True,exist_ok=True)
settings=dict(input=str(sd_input),out=str(SD_OUT),strength=STRENGTH,alpha=BLEND_ALPHA,
              steps=STEPS,cfg=CFG,seed=SD_SEED,prompt=PROMPT,negative_prompt=NEGATIVE_PROMPT)
(SD_OUT/'settings.json').write_text(json.dumps(settings,ensure_ascii=False,indent=2))
print('ภาพต้นทาง:',sd_input)


In [ ]:
# 8.3 สร้างภาพ: ครั้งแรกมีดาวน์โหลดโมเดลเพิ่ม
sd_worker = "import json,sys,time,math\nfrom pathlib import Path\nimport torch\nfrom PIL import Image\nfrom diffusers import StableDiffusionImg2ImgPipeline, AutoencoderKL, DPMSolverMultistepScheduler\nfrom huggingface_hub import model_info\ncfg=json.loads(Path(sys.argv[1]).read_text());out=Path(cfg['out'])\nstart=time.perf_counter()\nmodel_id='SG161222/Realistic_Vision_V5.1_noVAE'\nvae_id='stabilityai/sd-vae-ft-mse'\n# บันทึก revision ที่โหลดจริงเพื่อย้อนตรวจสอบและทำซ้ำ\nmodel_rev=model_info(model_id).sha\nvae_rev=model_info(vae_id).sha\nvae=AutoencoderKL.from_pretrained(vae_id,revision=vae_rev,torch_dtype=torch.float16)\npipe=StableDiffusionImg2ImgPipeline.from_pretrained(model_id,revision=model_rev,\n    vae=vae,torch_dtype=torch.float16,use_safetensors=True)\npipe.scheduler=DPMSolverMultistepScheduler.from_config(pipe.scheduler.config,\n    algorithm_type='dpmsolver++',use_karras_sigmas=True)\npipe.enable_model_cpu_offload()\npipe.enable_vae_tiling()\n# torch 2.x ใช้ scaled dot-product attention; ไม่ต้องติดตั้ง xformers\nsrc=Image.open(cfg['input']).convert('RGB')\nsrc.save(out/'01_hairfast_before.png')\ntorch.cuda.synchronize();load_sec=time.perf_counter()-start\nprint(f'โหลด/ดาวน์โหลดโมเดล: {load_sec:.2f} s',flush=True)\ngen=torch.Generator(device='cpu').manual_seed(cfg['seed'])\ntorch.cuda.reset_peak_memory_stats()\nt=torch.cuda.Event(enable_timing=True) # รายงาน wall-clock ด้านล่างรวม CPU offload ด้วย\nbegin=time.perf_counter()\nwith torch.inference_mode():\n    response=pipe(prompt=cfg['prompt'],negative_prompt=cfg['negative_prompt'],image=src,\n       strength=cfg['strength'],num_inference_steps=cfg['steps'],guidance_scale=cfg['cfg'],\n       generator=gen)\ntorch.cuda.synchronize();infer_sec=time.perf_counter()-begin\nif response.nsfw_content_detected and any(response.nsfw_content_detected):\n    raise RuntimeError('Pipeline flagged the output; no enhanced image was saved. Try another source image.')\nraw=response.images[0]\nassert raw.size==src.size, (raw.size,src.size)\npost_start=time.perf_counter()\nfinal=Image.blend(src,raw,cfg['alpha'])\nraw.save(out/'02_rv51_diffusion_raw.png')\nfinal.save(out/'03_rv51_preserved_final.png')\npost_sec=time.perf_counter()-post_start\ninfo={**cfg,'model_id':model_id,'model_revision':model_rev,'vae_id':vae_id,'vae_revision':vae_rev,\n      'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0),\n      'image_size':list(src.size),'effective_denoising_steps':int(cfg['steps']*cfg['strength']),\n      'model_download_load_seconds':load_sec,'diffusion_seconds':infer_sec,\n      'blend_and_save_seconds':post_sec,'diffusion_plus_postprocess_seconds':infer_sec+post_sec,\n      'worker_total_seconds':time.perf_counter()-start,\n      'peak_allocated_vram_gib':torch.cuda.max_memory_allocated()/1024**3}\n(out/'rv51_run_info.json').write_text(json.dumps(info,ensure_ascii=False,indent=2))\nprint(json.dumps(info,ensure_ascii=False,indent=2),flush=True)\n"
(BASE/'rv51_img2img.py').write_text(sd_worker)
run([SDPY,BASE/'rv51_img2img.py',SD_OUT/'settings.json'],log_name='diffusion_inference.log')


## 8.4 เปรียบเทียบและดูเวลา
แสดงภาพก่อน → diffusion โดยตรง → ผลสุดท้ายที่ผสมกับภาพเดิม ค่าเวลาวัดจริงแต่ละรอบ โดยแยกการโหลด/ดาวน์โหลดออกจากการปรับภาพ ไม่ใช่ผลวัดจากการทดสอบล่วงหน้า


In [ ]:
from PIL import ImageDraw
from IPython.display import display
assert (SD_OUT/'rv51_run_info.json').is_file(), 'รัน 8.3 ให้สำเร็จก่อน'
info=json.loads((SD_OUT/'rv51_run_info.json').read_text())
comparison=Image.new('RGB',(1536,570),'white')
draw=ImageDraw.Draw(comparison)
for i,(name,label) in enumerate([
 ('01_hairfast_before.png','Before: HairFastGAN'),
 ('02_rv51_diffusion_raw.png','Realistic Vision V5.1 (raw)'),
 ('03_rv51_preserved_final.png','After: blended final')]):
    with Image.open(SD_OUT/name) as im:
        thumbnail=ImageOps.contain(im.convert('RGB'),(500,500),Image.Resampling.LANCZOS)
    comparison.paste(thumbnail,(i*512+(512-thumbnail.width)//2,45))
    draw.text((i*512+12,15),label,fill='black')
comparison.save(SD_OUT/'04_before_after_comparison.png')
display(comparison)
print(f"โหลด/ดาวน์โหลดโมเดล: {info['model_download_load_seconds']:.2f} วินาที")
print(f"Diffusion: {info['diffusion_seconds']:.2f} วินาที")
print(f"ผสมภาพและบันทึก: {info['blend_and_save_seconds']:.2f} วินาที")
print(f"รวมปรับภาพ (ไม่รวมโหลด): {info['diffusion_plus_postprocess_seconds']:.2f} วินาที")
print(f"ทั้งหมดใน worker (รวมโหลด): {info['worker_total_seconds']:.2f} วินาที")
print('ขนาดผลลัพธ์:',info['image_size'],'| denoising steps:',info['effective_denoising_steps'])
print('ภาพสุดท้ายเต็มขนาด:')
display(Image.open(SD_OUT/'03_rv51_preserved_final.png'))


In [ ]:
# 8.5 ดาวน์โหลดผลสุดท้ายแยกรูป ไม่รวม ZIP
from google.colab import files
files.download(str(SD_OUT/'03_rv51_preserved_final.png'))


In [ ]:
# ตัวเลือกเพิ่มเติม: ดาวน์โหลดรูปใดก็เปลี่ยนชื่อด้านล่างแล้วรัน
DOWNLOAD_FILE = '04_before_after_comparison.png' #@param ['01_hairfast_before.png', '02_rv51_diffusion_raw.png', '04_before_after_comparison.png', 'rv51_run_info.json']
files.download(str(SD_OUT/DOWNLOAD_FILE))


### ข้อควรเข้าใจและแก้ปัญหา
- ความคมและความเป็นธรรมชาติเป็นเป้าหมายที่ต้องเทียบภาพจริง ไม่มีคะแนนยืนยันอัตโนมัติว่าคุณภาพดีขึ้น
- การผสมช่วยลดการเปลี่ยนแปลง แต่ไม่รับประกันเอกลักษณ์ และอาจทำให้เส้นผมซ้อน/ดูนุ่มหาก diffusion เปลี่ยนรูปทรงมาก ให้ลด strength ก่อน
- Prompt ไม่ระบุสีหรือทรงผมใหม่ เพื่อไม่ชี้นำให้เปลี่ยน; การรักษาลักษณะหลักมาจากภาพต้นทางและ strength ต่ำ
- การรัน 8.3 ซ้ำโหลดโมเดลใหม่ แต่ไฟล์น้ำหนักใช้ cache; รัน 8.2 ใหม่เพื่อสร้างโฟลเดอร์ทดลองใหม่ ไม่เขียนทับรอบก่อน
- ถ้า VRAM ไม่พอ ให้ปิดงาน GPU อื่นและเริ่ม runtime ใหม่; ไม่ลดความละเอียดให้อัตโนมัติเพราะจะทำให้ภาพเทียบไม่ตรงกัน
- หากภาพดำ/NaN หรือสีผิดปกติ อย่าใช้ผลนั้น; ส่ง `diffusion_inference.log` ให้ตรวจ VAE และ precision
- ส่วนเสริมนี้ตรวจ syntax และการเชื่อมต่อกับตัวแปรเดิมแล้ว แต่ยังไม่ได้ทดสอบ inference จริงบน Colab GPU ในสภาพแวดล้อมจัดทำไฟล์

แหล่งข้อมูล: [Realistic Vision V5.1 model card](https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE), [Diffusers img2img](https://huggingface.co/docs/diffusers/v0.32.2/en/using-diffusers/img2img), [VAE](https://huggingface.co/stabilityai/sd-vae-ft-mse)
